In [1]:
# from datasets import load_dataset
# from pathlib import Path

# DATASET_NAME = "lukaemon/mmlu"
# SUBJECT_CONFIG = "all"
# EVAL_SPLIT = "validation"
# MAX_EXAMPLES = None  # Set an integer for quick dry runs
# CHOICE_LABELS = ("A", "B", "C", "D")

# raw_dataset = load_dataset(DATASET_NAME, SUBJECT_CONFIG, split=EVAL_SPLIT)
# if MAX_EXAMPLES is not None:
#     raw_dataset = raw_dataset.select(range(min(MAX_EXAMPLES, len(raw_dataset))))

# def normalize_choices(entry):
#     choices = entry["choices"]
#     if isinstance(choices, dict):
#         ordered = [choices[label] for label in CHOICE_LABELS if label in choices]
#         return ordered or list(choices.values())
#     return list(choices)

# def normalize_gold(answer_value, num_choices):
#     if isinstance(answer_value, str):
#         candidate = answer_value.strip().upper()
#         if candidate and candidate[0] in CHOICE_LABELS[:num_choices]:
#             return candidate[0]
#         if candidate.isdigit():
#             answer_value = int(candidate)
#     if isinstance(answer_value, (int, float)):
#         idx = int(answer_value)
#         if 0 <= idx < num_choices:
#             return CHOICE_LABELS[idx]
#     raise ValueError(f"Unsupported answer format: {answer_value}")

# mmlu_examples = []
# for row in raw_dataset:
#     choices = normalize_choices(row)
#     gold = normalize_gold(row["answer"], len(choices))
#     mmlu_examples.append({
#         "question": row["question"],
#         "choices": choices,
#         "gold": gold,
#         "subject": row.get("subject", "unknown"),
#     })

# print(f"Prepared {len(mmlu_examples)} MMLU questions across {len(set(r['subject'] for r in mmlu_examples))} subjects.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
BATCH_SIZE = 8
GEN_KWARGS = {"max_new_tokens": 16, "do_sample": False}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
)

In [ ]:
def build_prompt(question: str, choices):
    formatted_choices = "\n".join(
        f"{label}. {choice}" for label, choice in zip(CHOICE_LABELS, choices)
    )
    return (
        "You are an expert multiple-choice tutor. Answer with the single best option letter.\n"
        f"Question: {question}\n"
        f"Choices:\n{formatted_choices}\n"
        "Answer:"
    )

def extract_choice(text: str):
    for char in text.upper():
        if char in CHOICE_LABELS:
            return char
    return None

def generate_batch(batch_examples):
    prompts = [build_prompt(item["question"], item["choices"]) for item in batch_examples]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )
    input_len = inputs["input_ids"].shape[1]
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            pad_token_id=tokenizer.pad_token_id,
            **GEN_KWARGS,
        )
    generated_tokens = outputs[:, input_len:]
    completions, predictions = [], []
    for idx in range(generated_tokens.size(0)):
        text = tokenizer.decode(generated_tokens[idx], skip_special_tokens=True).strip()
        completions.append(text)
        predictions.append(extract_choice(text))
    return completions, predictions

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

records = []
total_examples = len(mmlu_examples)

for start in tqdm(range(0, total_examples, BATCH_SIZE), desc="Evaluating"):
    batch = mmlu_examples[start : start + BATCH_SIZE]
    completions, predictions = generate_batch(batch)
    for example, completion, prediction in zip(batch, completions, predictions):
        records.append({
            "question": example["question"],
            "choices": example["choices"],
            "subject": example["subject"],
            "gold": example["gold"],
            "model_answer": completion,
            "prediction": prediction,
            "is_correct": prediction == example["gold"],
        })

results_df = pd.DataFrame(records)
results_df.head()

In [ ]:
overall_accuracy = results_df["is_correct"].mean()
per_subject_accuracy = (
    results_df.groupby("subject")["is_correct"].mean().sort_values(ascending=False)
)

print(f"Overall MMLU accuracy: {overall_accuracy:.2%}")
per_subject_accuracy.head(10)

In [ ]:
OUTPUT_PATH = Path("/home/eickhoff/esx208/RAG_Mech_Interp/RAG_best_practices/notebooks/mmlu_llama31_results.csv")
results_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(results_df)} rows to {OUTPUT_PATH}")
results_df.head()